# shearing_sheet_test — the byte-identity acceptance test

Runs the seeded 400-step shearing sheet and, if the C reference dump is present, verifies the two agree on every bit and prints both SHA-256 fingerprints. This is the project's headline acceptance test: 1,482 particles and ~102,500 collisions.

This notebook is self-contained: it builds the example with cargo, runs it, and shows the result. Everything it does can also be done by hand in a terminal:

```
cd rebound_rust
cargo build --release --example shearing_sheet_test
cd porttest
../target/release/examples/shearing_sheet_test 400
```

In [1]:
import os, subprocess
EXE = ".exe" if os.name == "nt" else ""   # platform executable suffix
NB_DIR  = os.getcwd()                       # <crate>/notebooks
ROOT    = os.path.dirname(os.path.dirname(NB_DIR))
CRATE   = os.path.join(ROOT, "rebound_rust")
WORK    = os.path.join(CRATE, "porttest")
if not os.path.exists(os.path.join(CRATE, "Cargo.toml")):
    raise SystemExit(
        "Could not find the crate. Run this notebook from "
        "the notebooks folder of a full checkout: " + CRATE)
# The example that is BUILT and RUN. It is usually the one the
# notebook is named after; where it differs (the stock
# shearing_sheet integrates forever by design) the terminating
# variant is used instead, and the note above says so.
EXAMPLE = "shearing_sheet_test"
OUTFILE = None
os.makedirs(WORK, exist_ok=True)
res = subprocess.run(["cargo", "build", "--release", "--example", EXAMPLE],
                     cwd=CRATE, capture_output=True, text=True)
print(res.stderr.strip()[-400:] or "build ok")

    Finished `release` profile [optimized] target(s) in 0.00s


In [2]:
import struct

def unbits(h):
    """Turn a 16-hex-digit IEEE-754 bit pattern back into a float."""
    return struct.unpack("<d", int(h, 16).to_bytes(8, "little"))[0]

def read_state(path):
    """Read one of the raw-bit state dumps into {label: [floats]}."""
    out = {}
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if not parts:
                continue
            key, rest = parts[0], parts[1:]
            vals = []
            for tok in rest:
                if len(tok) == 16:
                    try:
                        vals.append(unbits(tok))
                        continue
                    except ValueError:
                        pass
                vals.append(tok)
            out.setdefault(key, []).append(vals)
    return out

def compare(a, b, label_a="C", label_b="Rust"):
    """Byte-compare two dump files and report."""
    ta = open(a, "rb").read().replace(b"\r\n", b"\n")
    tb = open(b, "rb").read().replace(b"\r\n", b"\n")
    if ta == tb:
        print(f"BIT-IDENTICAL: {label_a} and {label_b} agree on every bit")
        return True
    print(f"MISMATCH between {label_a} and {label_b}")
    la, lb = ta.decode().splitlines(), tb.decode().splitlines()
    for i, (x, y) in enumerate(zip(la, lb)):
        if x != y:
            print(f"  line {i}:\n    {label_a}: {x}\n    {label_b}: {y}")
    return False


In [3]:
exe = os.path.join(CRATE, "target", "release", "examples", EXAMPLE + EXE)
res = subprocess.run([exe, "400"], cwd=WORK, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])

Toomre wavelength: 61.009899
N after init: 1482
final: t=1.91217633050229269e+04 steps=400 collisions=102478



In [4]:
import hashlib
def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest().upper()
c  = os.path.join(WORK, "state_c_final.txt")
rs = os.path.join(WORK, "state_rust_final.txt")
print("Rust SHA-256:", sha(rs))
if os.path.exists(c):
    print("C    SHA-256:", sha(c))
    compare(c, rs)
else:
    print("(C reference dump not present; run the C harness: porttest/problem_test 400 (Windows: rebound_test.exe) to compare)")


Rust SHA-256: 418C864DD1A610CBE8EA6D81ECAFA1E4CE6D36837494177D9875EE820EF0766F
(C reference dump not present; run the C harness: porttest/problem_test 400 (Windows: rebound_test.exe) to compare)
